# Exp 26: Coarse dropout, 5 folds

Card: [exp26_coarse_dropout_5fold.md](../../experiments/exp26_coarse_dropout_5fold.md)

In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import VIP, CVResult, Config, CrossValidator
from core.gradcam import GradCamFigures
from core.scores import Scores
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-coarse-dropout-5fold-smoke",
        "folds": [0, 1],
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SAMPLE = 2  # Cat, Dog images per comparison figure
    N_POSITIVES = 2  # images per VIP Grad-CAM figure
else:
    OVERRIDES = {}
    N_SAMPLE = 16  # exp19 sample
    N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.toml", **OVERRIDES)
EXP11 = Config.from_toml(CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.toml")
EXP16 = Config.from_toml(CONFIG_DIR / "exp16_downsampling" / "resnet18.toml")  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp26_coarse_dropout_5fold"
SAVE_FIGURES = not SMOKE
PROBS = [f"p_{c}" for c in CLASSES]

# dataviz reference palette, light mode
SERIES = {"exp11": "#eb6834", "coarse dropout": "#2a78d6"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Training

In [ ]:
results = {"exp11": CVResult.load(EXP11), "coarse dropout": CrossValidator(CONFIG).run()}

## Per-fold scores

In [ ]:
def fold_scores(result):
    """AP per class, mAP and Macro F1 (best-F1 thresholds tuned on same fold rows), one row per fold."""
    macro_f1 = {
        fold: Scores(part[CLASSES].to_numpy(), part[PROBS].to_numpy()).best_f1()["f1"].mean()
        for fold, part in result.oof.groupby("fold")
    }
    return result.fold_ap.assign(**{"Macro F1": pd.Series(macro_f1)})


scores = {name: fold_scores(r) for name, r in results.items()}
summary = pd.concat({name: pd.concat([s, s.agg(["mean", "std"])]) for name, s in scores.items()})
diff = (scores["coarse dropout"] - scores["exp11"]).dropna()
diff = pd.concat([diff, diff.agg(["mean"])])
display(summary.round(3), diff.round(3))

## Macro F1 on all out-of-fold rows, thresholds from other folds

In [ ]:
def cross_f1(result):
    """Per-class threshold (tuned on all rows) and F1 with each fold scored by thresholds tuned on rest."""
    oof = result.oof
    pred = pd.DataFrame(0, index=oof.index, columns=CLASSES)
    for fold in oof["fold"].unique():
        rest, part = oof[oof["fold"] != fold], oof["fold"] == fold
        thresholds = Scores(rest[CLASSES].to_numpy(), rest[PROBS].to_numpy()).best_f1()["threshold"]
        pred.loc[part] = (oof.loc[part, PROBS].to_numpy() >= thresholds.to_numpy()).astype(int)
    f1 = pd.Series(f1_score(oof[CLASSES], pred, average=None, zero_division=0), index=CLASSES)
    table = pd.DataFrame({"threshold": result.scores.best_f1()["threshold"], "f1": f1})
    table.loc["Macro F1"] = [float("nan"), f1.mean()]
    return table


pooled = pd.concat({name: cross_f1(r) for name, r in results.items()}, axis=1)
pooled.round(3)

## Confusion counts, VIP

In [ ]:
# best-F1 threshold per model and class on pooled OOF rows, as in submission
rows = []
for name, r in results.items():
    thresholds = r.scores.best_f1()["threshold"]
    for cls in VIP:
        y, pred = r.oof[cls].eq(1), r.oof[f"p_{cls}"] >= thresholds[cls]
        rows.append({
            "model": name, "class": cls,
            "TP": int((y & pred).sum()), "FP": int((~y & pred).sum()),
            "FN": int((y & ~pred).sum()),
        })
confusion = pd.DataFrame(rows).set_index(["class", "model"]).sort_index(level=0, sort_remaining=False)
confusion

## Precision-recall curves

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(recall, precision, color=SERIES[name], lw=2, drawstyle="steps-post", label=f"{name}, AP {average_precision_score(y, p):.3f}")
    ax.set(title=f"{cls} ({int(y.sum())} positives)", xlabel="recall", xlim=(0, 1.02), ylim=(0, 1.05))
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.suptitle("Out-of-fold rows of folds 0–4", color="#52514e")
fig.tight_layout()
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

## Grad-CAM Cat, Dog: exp11 vs coarse dropout

In [ ]:
cams = {"exp11": GradCamFigures(EXP11), "coarse dropout": GradCamFigures(CONFIG)}
source = cams["coarse dropout"] if SMOKE else GradCamFigures(EXP16)  # smoke OOF holds few rows
for cls in ["Cat", "Dog"]:
    rows = source.positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen images

In [ ]:
for cls in VIP:
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_validation.png"] = cams["coarse dropout"].positives(cls, n=N_POSITIVES)
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "full"}) as run:
    mlflow.log_table(summary.reset_index(names=["model", "fold"]), "fold_scores.json")
    mlflow.log_table(diff.reset_index(names="fold"), "fold_diff.json")
    flat = pooled.copy()
    flat.columns = [" ".join(c) for c in flat.columns]
    mlflow.log_table(flat.reset_index(names="class"), "pooled_f1.json")
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)